[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-2-generative-ai/adv-00-token-costs-and-spend-guard.ipynb)

# Advanced Discussion 0 — know the bill before you run

Token counts across text types and languages, cost projection for a RAG question and an evaluation run, and a spend guard tested on a runaway loop. Prices are placeholders.

**Optional advanced-discussion lab** for Generative AI and Large Language Models with Python (AIBits Academy). Not graded and not needed for the certificate. Everything here runs on the Colab free tier; CPU is enough unless a note says otherwise.

Pure counting and arithmetic; runs in seconds.

In [ ]:
%pip install -q tiktoken numpy

In [ ]:
import tiktoken, math, numpy as np
enc = tiktoken.get_encoding("cl100k_base")
IN, OUT = 3.00, 15.00                    # PLACEHOLDER dollars per million tokens; read your provider's current price list
def usd(inp, out): return (inp * IN + out * OUT) / 1e6

## 1. tokens are not words: the same amount of meaning costs different amounts in different kinds of text

In [ ]:
samples = {
 "English prose": "The shipment was delayed because the carrier rerouted the truck through the port, so the customer should expect it on Thursday.",
 "Python code": "def total(items):\n    return sum(i['price'] * i['qty'] for i in items if i['qty'] > 0)\n",
 "JSON": '{"order_id": "ORD-004471", "items": [{"sku": "A-100", "qty": 2, "price": 19.99}], "status": "shipped"}',
 "Numbers": "4471 8820 1093 5567 2210 9984 3301 7745 6002 1189 4420 9053",
 "Hindi (Devanagari)": "शिपमेंट में देरी हुई क्योंकि वाहक ने ट्रक को बंदरगाह के रास्ते भेजा, इसलिए ग्राहक को गुरुवार तक इंतज़ार करना चाहिए।",
 "Tamil": "கப்பல் தாமதமானது, ஏனெனில் கேரியர் லாரியை துறைமுகம் வழியாக மாற்றியது, எனவே வாடிக்கையாளர் வியாழக்கிழமை வரை எதிர்பார்க்க வேண்டும்.",
 "German": "Die Sendung verzögerte sich, weil der Spediteur den Lastwagen über den Hafen umgeleitet hat, daher sollte der Kunde sie am Donnerstag erwarten."}
print("text type            characters   words   tokens   tokens per word   tokens per 100 characters")
base = None
for k, t in samples.items():
    n = len(enc.encode(t)); w = len(t.split()); print("%-20s %7d    %5d   %6d      %6.2f            %6.1f" % (k, len(t), w, n, n / w, 100 * n / len(t)))

## 2. what one lab run costs: multiply, do not guess

In [ ]:
sys_prompt = "You are a careful support assistant. Answer only from the context. Cite the chunk id. If the context does not contain the answer, say so. " * 3
chunk = ("Refund requests above 500 need approval from a manager. " * 40)          # ~one retrieval chunk
q = "What is the refund limit that needs approval?"
p_in = len(enc.encode(sys_prompt)) + 5 * len(enc.encode(chunk)) + len(enc.encode(q)); p_out = 120
print("\none RAG question: system prompt + 5 chunks + question = %d input tokens; assume %d output tokens -> $%.5f" % (p_in, p_out, usd(p_in, p_out)))
for calls in (20, 100, 500, 2000):
    print("  %5d calls: $%.2f" % (calls, calls * usd(p_in, p_out)))
print("an evaluation over 300 golden questions with a judge call each (judge sees question + answer + reference = ~600 tokens in, 80 out): $%.2f" % (300 * (usd(p_in, p_out) + usd(600, 80))))
print("the same run with 10 chunks instead of 5: $%.2f (+%.0f%%)" % (300 * (usd(p_in + 5 * len(enc.encode(chunk)), p_out) + usd(600, 80)), 100 * (usd(p_in + 5 * len(enc.encode(chunk)), p_out) + usd(600, 80)) / (usd(p_in, p_out) + usd(600, 80)) - 100))

## 3. a spend guard, and the runaway it prevents

In [ ]:
class BudgetExceeded(Exception): pass
class Guard:
    def __init__(s, max_usd, max_calls): s.max_usd, s.max_calls, s.spent, s.calls = max_usd, max_calls, 0.0, 0
    def charge(s, inp, out):
        cost = usd(inp, out)
        if s.calls + 1 > s.max_calls or s.spent + cost > s.max_usd: raise BudgetExceeded("stopped after %d calls, $%.4f spent" % (s.calls, s.spent))
        s.calls += 1; s.spent += cost
def buggy_loop(guard=None, max_iter=100000):
    """A loop whose stop condition never triggers (a typo in the exit test): each turn re-sends a growing history."""
    hist = 500; total = 0.0; n = 0
    try:
        while n < max_iter:
            if guard: guard.charge(hist, 100)
            total += usd(hist, 100); hist += 300; n += 1
    except BudgetExceeded as e: return total, n, str(e)
    return total, n, "ran to the iteration limit"
t, n, msg = buggy_loop(max_iter=2000); print("\nrunaway loop with no guard, cut off at 2,000 iterations: %d calls, $%.2f (history reached %d tokens)" % (n, t, 500 + 300 * n))
t, n, msg = buggy_loop(Guard(max_usd=1.00, max_calls=200)); print("same loop with Guard(max $1.00, max 200 calls): %d calls, $%.2f -- %s" % (n, t, msg))
print("note the cost is quadratic in turns: each turn resends the whole history, so doubling the turns roughly quadruples the bill")

## Your turn

Change one setting at a time (seed, learning rate, ε, perplexity, threshold), re-run, and write one sentence about what changed and why. Keep the numbers: your results will differ slightly from the chapter's, and that difference is worth explaining.